# ARC Prize 2026: CGR development scaffold (ARC-AGI-2)

**Status: prototype, not a verified Kaggle submission and not a full implementation of the paper.**

**Licence.** The ARC submission artifacts in this repository — this notebook, `ARC_PRIZE_2026_KAGGLE_PLAN.md` and `results/arc_kaggle_scaffold_smoke/` — are released under **MIT No Attribution (MIT-0)**, which is the licence ARC Prize 2026 names for authored code. See `LICENSE-MIT-0.txt`. The rest of the repository remains under MIT.

**What this is.** An offline, executable ARC-AGI-2 submission builder plus two optional Qwen diagnostics (KV-cache crop coherence and coder-adapter behaviour). The ARC candidate selector applies deterministic training-example checks over seven global transforms. It is **not** the trained arithmetic localizer, the KV-cache repair loop, or the specialist router.

**ARC accuracy is unmeasured.** The seven-transform family is limited, and training fit does not guarantee test accuracy. Partial-fit fallbacks also produce predictions; training-set coverage is not a proved upper bound. These predictions do not demonstrate the paper's learned repair mechanism.

**Evidence correction:** historical arithmetic cache and full-loop speedup interpretations are withdrawn. This notebook's optional coherence diagnostic is not a timing benchmark.

**Saved outputs**, when present, come from a local CPU run in `demo` mode; they are not a Kaggle execution and not an ARC score. The optional model diagnostics are skipped there because no checkpoint is attached — their recorded run, with the checkpoint and adapter, is in `results/arc_kaggle_scaffold_smoke/`.

## 1. Configure the run

`MODE` is inferred: it becomes `competition` when `/kaggle/input` exists, otherwise `demo`. Override with `CGR_ARC_MODE`.

| Variable | Meaning |
|---|---|
| `CGR_ARC_MODE` | `demo` or `competition` (auto-detected by default) |
| `ARC_CHALLENGES_PATH` | explicit test-challenges JSON when autodiscovery is ambiguous |
| `CGR_LIVE_QWEN=1` + `CGR_QWEN_MODEL_DIR` | run the optional cache diagnostic from an attached snapshot |
| `CGR_LIVE_ADAPTER=1` + `CGR_CODER_ADAPTER_DIR` | additionally run the coder diagnostic |
| `CGR_ADAPTER_LICENCE_ACK=1` | required before the coder diagnostic will load the adapter |
| `CGR_ARC_STRICT` | `1` (default in demo) re-raises per-task errors instead of falling back |

The coder adapter is a local derivative whose training-data provenance is unresolved. It is excluded from any public ARC submission until that is documented; see the licence inventory in the plan.

In [ ]:
import hashlib, json, os, re, time
from pathlib import Path

NOTEBOOK_LICENSE = "MIT-0"

KAGGLE_INPUT = Path("/kaggle/input")
MODE = os.environ.get("CGR_ARC_MODE") or ("competition" if KAGGLE_INPUT.is_dir() else "demo")
LIVE_QWEN = os.environ.get("CGR_LIVE_QWEN", "0") == "1"
LIVE_ADAPTER = os.environ.get("CGR_LIVE_ADAPTER", "0") == "1"
STRICT = os.environ.get("CGR_ARC_STRICT", "1" if MODE == "demo" else "0") == "1"
MODEL_DIR = os.environ.get("CGR_QWEN_MODEL_DIR", "")
ADAPTER_DIR = os.environ.get("CGR_CODER_ADAPTER_DIR", "")
SAMPLE_SUBMISSION = os.environ.get("ARC_SAMPLE_SUBMISSION", "")
EXPECTED_QWEN_REVISION = "c1899de289a04d12100db370d81485cdf75e47ca"
CACHE_API_VALIDATED_ON = "4.57.6"

if MODE not in {"demo", "competition"}:
    raise ValueError("CGR_ARC_MODE must be demo or competition")
if LIVE_ADAPTER and not LIVE_QWEN:
    raise ValueError("CGR_LIVE_ADAPTER requires CGR_LIVE_QWEN=1")
os.environ["HF_HUB_OFFLINE"] = "1"
os.environ["TRANSFORMERS_OFFLINE"] = "1"
print(f"licence={NOTEBOOK_LICENSE} | mode={MODE} | strict={STRICT} | "
      f"qwen_diagnostic={LIVE_QWEN} | adapter_diagnostic={LIVE_ADAPTER}")

In [ ]:
MAX_SIDE = 30
MIN_SIDE = 1

def validate_grid(grid):
    if not isinstance(grid, list) or not MIN_SIDE <= len(grid) <= MAX_SIDE:
        raise ValueError(f"Grid must contain {MIN_SIDE}..{MAX_SIDE} rows")
    if not isinstance(grid[0], list) or not MIN_SIDE <= len(grid[0]) <= MAX_SIDE:
        raise ValueError(f"Grid must contain {MIN_SIDE}..{MAX_SIDE} columns")
    width = len(grid[0])
    if any(not isinstance(row, list) or len(row) != width for row in grid):
        raise ValueError("Grid rows must be rectangular")
    if any(type(value) is not int or not 0 <= value <= 9 for row in grid for value in row):
        raise ValueError("Grid values must be integers in 0..9")
    return grid

def identity(g): return [row[:] for row in g]
def flip_h(g): return [row[::-1] for row in g]
def flip_v(g): return [row[:] for row in g[::-1]]
def rot180(g): return [row[::-1] for row in g[::-1]]
def transpose(g): return [list(row) for row in zip(*g)]
def rot90(g): return [list(row) for row in zip(*g[::-1])]
def rot270(g): return [list(row) for row in zip(*g)][::-1]

HYPOTHESES = [("identity", identity), ("flip_h", flip_h), ("flip_v", flip_v),
              ("rot180", rot180), ("transpose", transpose), ("rot90", rot90),
              ("rot270", rot270)]

def rank_candidates(train):
    # Score every hypothesis on the public training pairs; best partial fit first.
    checks = []
    for name, transform in HYPOTHESES:
        results = [transform(pair["input"]) == pair["output"] for pair in train]
        checks.append({"candidate": name,
                       "training_examples_passed": sum(results),
                       "training_examples": len(results),
                       "accepted": all(results)})
    order = sorted(range(len(HYPOTHESES)),
                   key=lambda i: (-checks[i]["training_examples_passed"], i))
    return checks, order

def solve_task(task):
    train, test = task.get("train"), task.get("test")
    if not isinstance(train, list) or not train or not isinstance(test, list) or not test:
        raise ValueError("Each task must contain nonempty train and test lists")
    for pair in train:
        validate_grid(pair["input"]); validate_grid(pair["output"])
    for pair in test:
        validate_grid(pair["input"])  # Never inspect test outputs.

    checks, order = rank_candidates(train)
    accepted = [c["candidate"] for c in checks if c["accepted"]]
    predicted, sources, distinct = [], [], []
    for pair in test:
        grids, names = [], []
        for i in order:
            grid = validate_grid(HYPOTHESES[i][1](pair["input"]))
            if grid not in grids:                 # keep attempt_2 distinct from attempt_1
                grids.append(grid); names.append(HYPOTHESES[i][0])
            if len(grids) == 2:
                break
        if len(grids) < 2:                        # only when every transform agrees
            grids.append([row[:] for row in grids[0]])
            names.append(names[0] + "_duplicate")
        predicted.append({"attempt_1": grids[0], "attempt_2": grids[1]})
        sources.append(names)
        distinct.append(grids[0] != grids[1])
    return predicted, {"checks": checks,
                       "accepted": accepted,
                       "selected": sources[0],
                       "selected_by_test": sources,
                       "distinct_attempts": distinct,
                       "fallback": not accepted}

def identity_submission(task):
    entries = []
    for pair in task["test"]:
        grid = [row[:] for row in validate_grid(pair["input"])]
        entries.append({"attempt_1": grid, "attempt_2": [row[:] for row in grid]})
    return entries

def create_submission(challenges, strict=None):
    # Build {task_id: [{"attempt_1": grid, "attempt_2": grid}, ...]} for every test input.
    if not isinstance(challenges, dict) or not challenges:
        raise ValueError("Challenge file must be a nonempty mapping")
    strict = STRICT if strict is None else strict
    predictions, logs = {}, {}
    for task_id, task in challenges.items():
        if not isinstance(task_id, str) or not task_id:
            raise ValueError("Task ID must be a nonempty string")
        try:
            predictions[task_id], logs[task_id] = solve_task(task)
        except Exception as exc:
            if strict:
                raise
            predictions[task_id] = identity_submission(task)
            logs[task_id] = {"error": f"{type(exc).__name__}: {exc}",
                             "fallback": True,
                             "distinct_attempts": [False] * len(predictions[task_id]),
                             "selected": ["identity_error"] * len(predictions[task_id])}
    if set(predictions) != set(challenges):
        raise ValueError("Prediction task IDs do not match the challenge task IDs")
    for task_id, entries in predictions.items():
        if len(entries) != len(challenges[task_id]["test"]):
            raise ValueError(f"{task_id}: expected one entry per test input")
        for entry in entries:
            if set(entry) != {"attempt_1", "attempt_2"}:
                raise ValueError(f"{task_id}: entry must have attempt_1 and attempt_2")
            validate_grid(entry["attempt_1"]); validate_grid(entry["attempt_2"])
    return predictions, logs

In [ ]:
toy = {"toy_flip": {"train": [
    {"input": [[1, 0, 2], [3, 4, 5]], "output": [[2, 0, 1], [5, 4, 3]]},
    {"input": [[6, 7, 8]], "output": [[8, 7, 6]]}],
    "test": [{"input": [[1, 2, 3], [4, 5, 6]]}]}}
toy_predictions, toy_log = create_submission(toy)
assert toy_log["toy_flip"]["checks"][0]["accepted"] is False
assert toy_log["toy_flip"]["checks"][1]["accepted"] is True, "expected flip_h to be accepted"
assert toy_predictions["toy_flip"][0]["attempt_1"] == [[3, 2, 1], [6, 5, 4]]
assert toy_predictions["toy_flip"][0]["attempt_2"] != toy_predictions["toy_flip"][0]["attempt_1"], \
    "attempt_2 must be a distinct hypothesis, not a copy of attempt_1"
assert toy_log["toy_flip"]["distinct_attempts"] == [True]
print("Synthetic training checks:", json.dumps(toy_log["toy_flip"], indent=2))
print("Synthetic test prediction:", toy_predictions["toy_flip"])

In [ ]:
if LIVE_QWEN:
    import torch, transformers
    from transformers import AutoModelForCausalLM, AutoTokenizer

    if transformers.__version__ != CACHE_API_VALIDATED_ON:
        print(f"[warn] the crop API was validated on transformers=={CACHE_API_VALIDATED_ON}; "
              f"running {transformers.__version__}. Capability is checked at runtime.")
    if not MODEL_DIR or not Path(MODEL_DIR).is_dir():
        raise FileNotFoundError("Attach the Qwen snapshot and set CGR_QWEN_MODEL_DIR")

    model_path = Path(MODEL_DIR).resolve()
    snapshot_revision = model_path.name if re.fullmatch(r"[0-9a-f]{40}", model_path.name) else None
    revision_path_matches = snapshot_revision == EXPECTED_QWEN_REVISION
    revision_verified = False  # A directory name is not weight-content verification.
    if snapshot_revision is None:
        print("[warn] model directory is not an HF snapshot hash; revision cannot be verified from the path.")
    elif not revision_path_matches:
        raise RuntimeError(f"Attached snapshot is {snapshot_revision}, expected {EXPECTED_QWEN_REVISION}")

    device = "cuda" if torch.cuda.is_available() else "cpu"
    dtype = torch.float16 if device == "cuda" else torch.float32
    tokenizer = AutoTokenizer.from_pretrained(model_path, local_files_only=True)
    try:
        model = AutoModelForCausalLM.from_pretrained(model_path, dtype=dtype, local_files_only=True)
    except TypeError:  # transformers < 4.56 uses torch_dtype
        model = AutoModelForCausalLM.from_pretrained(model_path, torch_dtype=dtype, local_files_only=True)
    model = model.to(device).eval()

    ids = tokenizer("A sequence of checks: first preserve the valid prefix; then recompute the suffix.",
                    return_tensors="pt")["input_ids"].to(device)
    if ids.shape[1] < 4:
        raise ValueError("Diagnostic prompt is unexpectedly short")
    boundary = ids.shape[1] // 2
    tol = 0.125 if dtype == torch.float16 else 1e-4
    tolerance_validated = dtype != torch.float16

    with torch.inference_mode():
        full = model(input_ids=ids, use_cache=True, return_dict=True)
        cache = full.past_key_values
        if not hasattr(cache, "crop"):
            raise TypeError(f"{type(cache).__name__} does not support absolute-length crop")
        cache.crop(boundary)
        if cache.get_seq_length() != boundary:
            raise AssertionError("Unexpected cache length after crop")
        replay = model(input_ids=ids[:, boundary:], past_key_values=cache,
                       attention_mask=torch.ones_like(ids), use_cache=True, return_dict=True)
        delta = (replay.logits[:, -1].float() - full.logits[:, -1].float()).abs().max().item()
        argmax_equal = (replay.logits[:, -1].argmax(-1).item()
                        == full.logits[:, -1].argmax(-1).item())

    cache_report = {"device": device, "dtype": str(dtype),
                    "torch": torch.__version__,
                    "transformers": transformers.__version__,
                    "expected_base_revision": EXPECTED_QWEN_REVISION,
                    "revision_verified": revision_verified, "revision_path_matches": revision_path_matches,
                    "crop_boundary": boundary, "prompt_tokens": int(ids.shape[1]),
                    "max_abs_logit_delta": delta, "tolerance": tol,
                    "tolerance_validated": tolerance_validated,
                    "within_tolerance": delta <= tol, "argmax_match": argmax_equal}
    print(json.dumps(cache_report, indent=2))
    if not tolerance_validated:
        print("[warn] the fp16 tolerance has not been validated on GPU; treat it as a diagnostic, not a gate.")
    assert argmax_equal, "cropped-cache continuation changed the next-token argmax"
else:
    print("Live Qwen cache diagnostic skipped; enable only with an attached checkpoint.")

In [ ]:
if LIVE_ADAPTER:
    import subprocess, sys
    from peft import PeftModel

    print("[licence] The coder adapter is a local derivative. Its training-data provenance is")
    print("[licence] unresolved (see the licence inventory in ARC_PRIZE_2026_KAGGLE_PLAN.md).")
    print("[licence] Do not attach it to a public ARC submission until that is documented.")
    if os.environ.get("CGR_ADAPTER_LICENCE_ACK", "0") != "1":
        raise RuntimeError("Set CGR_ADAPTER_LICENCE_ACK=1 after reviewing the adapter's provenance.")

    if not ADAPTER_DIR or not Path(ADAPTER_DIR).is_dir():
        raise FileNotFoundError("Attach the coder adapter and set CGR_CODER_ADAPTER_DIR")
    adapter_cfg = json.loads((Path(ADAPTER_DIR) / "adapter_config.json").read_text(encoding="utf-8"))
    adapter_base = adapter_cfg.get("base_model_name_or_path")
    if adapter_base != "Qwen/Qwen3-0.6B":
        raise ValueError(f"Adapter base model {adapter_base!r} does not match the expected backbone")
    model = PeftModel.from_pretrained(model, ADAPTER_DIR, is_trainable=False).eval()

    messages = [{"role": "user", "content": "Write only Python code for factorial(n). Return n factorial for a nonnegative integer n. No imports or examples."}]
    text = tokenizer.apply_chat_template(messages, tokenize=False, add_generation_prompt=True, enable_thinking=False)
    inputs = tokenizer(text, return_tensors="pt", add_special_tokens=False).to(device)

    def answer():
        with torch.inference_mode():
            output = model.generate(**inputs, max_new_tokens=160, do_sample=False,
                                    pad_token_id=tokenizer.pad_token_id)
        generated = output[0, inputs["input_ids"].shape[-1]:]
        return {"text": tokenizer.decode(generated, skip_special_tokens=True).strip(),
                "eos": bool(generated.numel() and generated[-1].item() == tokenizer.eos_token_id)}

    verifier = r'''
import ast, json, re, sys
try:
 text = json.loads(sys.stdin.read())["text"]
 match = re.search(r"```(?:python)?\s*\n(.*?)```", text, re.S | re.I)
 source = match.group(1) if match else text.strip()
 tree = ast.parse(source)
 if any(not isinstance(n, ast.FunctionDef) or n.decorator_list for n in tree.body): raise ValueError("function definitions only")
 for node in ast.walk(tree):
  if isinstance(node, (ast.Import, ast.ImportFrom, ast.Global, ast.Nonlocal, ast.ClassDef, ast.AsyncFunctionDef, ast.Attribute)): raise ValueError("unsupported operation")
  if isinstance(node, ast.Name) and node.id.startswith("__"): raise ValueError("dunder name")
 namespace = {"__builtins__": {"range": range, "len": len, "int": int}}
 exec(compile(tree, "<candidate>", "exec"), namespace)
 fn = namespace.get("factorial")
 if not callable(fn): raise ValueError("factorial missing")
 cases = [(0, 1), (1, 1), (5, 120)]
 results = [{"input": n, "expected": expected, "actual": fn(n)} for n, expected in cases]
 for r in results: r["passed"] = type(r["actual"]) is int and r["actual"] == r["expected"]
 print(json.dumps({"passed": all(r["passed"] for r in results), "cases": results}))
except Exception as exc:
 print(json.dumps({"passed": False, "reason": type(exc).__name__ + ": " + str(exc)}))
'''

    def verify(candidate):
        if not candidate["eos"]:
            return {"passed": False, "reason": "generation reached token cap"}
        try:
            proc = subprocess.run([sys.executable, "-I", "-c", verifier], input=json.dumps(candidate),
                                  text=True, capture_output=True, timeout=3, check=False)
            return json.loads(proc.stdout) if proc.returncode == 0 else {"passed": False, "reason": "verifier subprocess failed"}
        except (subprocess.TimeoutExpired, ValueError) as exc:
            return {"passed": False, "reason": type(exc).__name__}

    with model.disable_adapter():
        base_answer = answer()
    model.base_model.enable_adapter_layers()
    adapter_answer = answer()
    active = model.active_adapters
    print("Adapters active:", active() if callable(active) else active)
    print("BASE:", base_answer["text"], "\nBASE CHECK:", verify(base_answer))
    print("CODER:", adapter_answer["text"], "\nCODER CHECK:", verify(adapter_answer))
else:
    print("Live coder diagnostic skipped; no adapter result was generated.")

In [ ]:
if MODE == "competition":
    explicit = os.environ.get("ARC_CHALLENGES_PATH", "")
    if explicit:
        challenge_path = Path(explicit)
    else:
        matches = sorted(KAGGLE_INPUT.rglob("*test*challenges*.json"))
        if len(matches) != 1:
            raise RuntimeError(f"Expected one mounted test challenges file; found {len(matches)}. "
                               "Set ARC_CHALLENGES_PATH explicitly.")
        challenge_path = matches[0]
    if not challenge_path.is_file():
        raise FileNotFoundError(challenge_path)
    raw = challenge_path.read_bytes()
    challenges = json.loads(raw)
    target = Path("/kaggle/working/submission.json")
else:
    if "toy" not in globals():
        raise RuntimeError("Run the synthetic-task cell first, or set CGR_ARC_MODE=competition.")
    challenges = toy
    target = Path(os.environ.get("CGR_ARC_DEMO_DIR", ".")) / "arc_demo_predictions.json"

# Write a complete, valid file before solving. A missing submission scores 0 with
# no diagnostic, so the run must always leave a parseable artifact behind.
target.parent.mkdir(parents=True, exist_ok=True)
placeholder = {task_id: identity_submission(task) for task_id, task in challenges.items()}
target.write_text(json.dumps(placeholder, separators=(",", ":")), encoding="utf-8")

start = time.perf_counter()
submission, logs = create_submission(challenges)
target.write_text(json.dumps(submission, separators=(",", ":")), encoding="utf-8")
if json.loads(target.read_text(encoding="utf-8")) != submission:
    raise AssertionError("written submission does not round-trip")

sample_path = Path(SAMPLE_SUBMISSION) if SAMPLE_SUBMISSION else None
if sample_path is None:
    candidates = sorted(KAGGLE_INPUT.rglob("sample_submission.json"))
    sample_path = candidates[0] if len(candidates) == 1 else None
if sample_path and sample_path.is_file():
    sample = json.loads(sample_path.read_text(encoding="utf-8"))
    missing = sorted(set(sample) - set(submission))
    extra = sorted(set(submission) - set(sample))
    print(f"Sample reconciliation: {len(sample)} sample tasks, {len(missing)} missing, {len(extra)} extra")
    if missing:
        print("  MISSING:", missing[:10])
    if extra:
        print("  EXTRA:", extra[:10])
else:
    print("Sample reconciliation skipped; no unambiguous sample_submission.json found.")

test_inputs = sum(len(entries) for entries in submission.values())
distinct = sum(sum(1 for d in log.get("distinct_attempts", []) if d) for log in logs.values())
errors = {k: v["error"] for k, v in logs.items() if "error" in v}
print("Wrote", target, "| tasks:", len(submission), "| test inputs:", test_inputs,
      "| elapsed seconds:", round(time.perf_counter() - start, 3))
print(f"Distinct attempt pairs: {distinct}/{test_inputs} | tasks without a fully accepted candidate: "
      f"{sum(1 for v in logs.values() if v.get('fallback'))}/{len(logs)}")
if errors:
    print("Task errors (fell back to identity):", json.dumps(errors, indent=2))
if MODE == "demo":
    print("Synthetic output only. This is not a Kaggle submission or ARC score.")

## 2. What this run does and does not establish

It establishes that the submission builder emits the ARC-2 schema — one entry per test input, exactly `attempt_1` and `attempt_2`, grids validated as rectangular integers in 0..9 with sides 1..30 — and that `attempt_2` is a distinct hypothesis wherever the transform family can produce one.

It does not establish an ARC score, a Kaggle execution, or any part of the paper's mechanism. The seven global transforms are a symbolic baseline; a trained ARC error localizer, a cache-repair loop and a specialist router are all absent from the prediction path.